In [4]:
import pandas as pd

### Câu 1:Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

In [5]:

orders= pd.read_csv('../data/Transaction/orders.csv', parse_dates=['order_date'])
if 'customer_id' not in orders.columns or 'order_date' not in orders.columns:
    raise SystemExit('orders.csv must contain customer_id and order_date columns')

# sx và tính khoảng cách giữa các đơn hàng liên tiếp của cùng một khách hàng
orders = orders.sort_values(['customer_id', 'order_date'])
orders['prev_order_date'] = orders.groupby('customer_id')['order_date'].shift(1)
orders['inter_order_days'] = (orders['order_date'] - orders['prev_order_date']).dt.days

# chỉ giữ lại các khách hàng có nhiều hơn một đơn hàng (có khoảng cách giữa các đơn hàng)
gaps = orders.dropna(subset=['inter_order_days'])

median_gap = gaps['inter_order_days'].median()

print(int(median_gap),round(median_gap, 2),sep="\n")



144
144.0


### Câu 2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?

In [6]:
products = pd.read_csv('../data/Master/products.csv')
#loại bỏ trường hop price = 0
products = products[products['price'] != 0]
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']
res = products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
top = res.index[0]
print(top, float(res.iloc[0]))

Standard 0.31344174843884803


### Câu 3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [7]:
returns=pd.read_csv('../data/Transaction/returns.csv', parse_dates=['return_date'])
res=returns.merge(products[products['segment'] == 'Streetwear'][['product_id']], on='product_id', how='left')
reason_counts = res['return_reason'].value_counts()
most_common_reason = reason_counts.idxmax()
print(most_common_reason)

wrong_size


### Câu 4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [8]:
web_traffic = pd.read_csv('../data/Operational/web_traffic.csv', parse_dates=['date'])
bounce_rates = web_traffic.groupby('traffic_source')['bounce_rate'].mean()
bounce_rates.sort_values(inplace=True,ascending=True)
bounce_rates.idxmin()

'email_campaign'

### 5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

In [9]:
order_items = pd.read_csv('../data/Transaction/order_items.csv',low_memory=False)
total_rows = len(order_items)
promo_rows = order_items['promo_id'].notna().sum()
promo_percentage = (promo_rows / total_rows) * 100
print(round(promo_percentage, 2))

38.66


### Cau 6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

In [10]:
customers = pd.read_csv('../data/Master/customers.csv')
orders = pd.read_csv('../data/Transaction/orders.csv')
# lọc khách hàng có age_group khác null
customers = customers[customers['age_group'].notna()]
# đếm số đơn hàng của mỗi khách hàng
order_counts = orders['customer_id'].value_counts()
# gộp số đơn hàng vào dữ liệu khách hàng
customers = customers.merge(order_counts.rename('order_count'), left_on='customer_id', right_index=True, how='left')
customers['order_count'] = customers['order_count'].fillna(0)
# tính số khách hàng và tổng số đơn hàng theo nhóm tuổi
age_group_stats = customers.groupby('age_group').agg({'customer_id': 'count', 'order_count': 'sum'})
age_group_stats['avg_orders_per_customer'] = age_group_stats['order_count'] / age_group_stats['customer_id']
# tìm nhóm tuổi có số đơn hàng trung bình cao nhất
top_age_group = age_group_stats['avg_orders_per_customer'].idxmax()
print(top_age_group, age_group_stats.loc[top_age_group, 'avg_orders_per_customer'])

55+ 5.406851452775507


### Cau 7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?

In [11]:
geography = pd.read_csv('../data/Master/geography.csv')
sales = pd.read_csv('../data/Analytical/sales.csv')
orders = pd.read_csv('../data/Transaction/orders.csv', parse_dates=['order_date'])
payments = pd.read_csv('../data/Transaction/payments.csv')

ord_geo = orders.merge(geography[['zip','region']], on='zip', how='left')
ord_pay = ord_geo.merge(payments[['order_id','payment_value']], on='order_id', how='left')
region_rev = ord_pay.groupby('region')['payment_value'].sum().sort_values(ascending=False)
top = region_rev.index[0]
print(region_rev)
print("Đáp án: ",top, float(region_rev.iloc[0]))

region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: payment_value, dtype: float64
Đáp án:  East 7291150819.12


### Cau 8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?

In [12]:
orders = pd.read_csv('../data/Transaction/orders.csv')
payments = pd.read_csv('../data/Transaction/payments.csv')
cancelled = orders[orders['order_status']=='cancelled']

merged = cancelled.merge(
    payments[['order_id','payment_method']],
    on='order_id',
    how='left',
    suffixes=('', '_pay')
)

top = merged['payment_method'].value_counts().idxmax()
print(top)

credit_card


### Cau 9.Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?


In [13]:

returns = pd.read_csv('../data/Transaction/returns.csv')
order_items = pd.read_csv('../data/Transaction/order_items.csv',low_memory=False)
products = pd.read_csv('../data/Master/products.csv')
ret = returns.merge(products[['product_id','size']], on='product_id', how='left')
oi = order_items.merge(products[['product_id','size']], on='product_id', how='left')
sizes = ['S','M','L','XL']
rates = {}
for s in sizes:
    r = ret[ret['size']==s].shape[0]
    denom = oi[oi['size']==s].shape[0]
    rates[s] = (r/denom if denom>0 else float('nan'))
#tìm kích thước có tỷ lệ trả hàng cao nhất, bỏ qua các kích thước có tỷ lệ không xác định (nan)
best = max(rates.items(), key=lambda x: (float('-inf') if pd.isna(x[1]) else x[1]))
print(rates, best, sep="\n")

{'S': 0.05651526952720847, 'M': 0.05566009930396536, 'L': 0.05624978345479113, 'XL': 0.05520010361352157}
('S', 0.05651526952720847)


### Câu 10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

In [14]:
payments = pd.read_csv('../data/Transaction/payments.csv')
plan_revenue = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
top_plan = plan_revenue.index[0]
print(top_plan, float(plan_revenue.iloc[0]))

6 24446.65440296606
